In [9]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
import numpy as np
import re
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [ ]:
original_train = pd.read_csv("Data/train.csv")
original_test = pd.read_csv("Data/test.csv", names=list(original_train.columns), header=0)

train_df = original_train.copy()
test_df = original_test.copy()

assert list(train_df.columns) == list(test_df.columns)

In [11]:
# Rename columns in-place so that they only have letters, digits, and underscores:
train_df.columns = [
    re.sub(r"[^A-Za-z0-9_]+", "_", col)  # Replace non-alphanumeric/underscore chars with "_"
    for col in train_df.columns
]



X = train_df.drop(columns="Activity")
y = train_df["Activity"]
X_test = test_df.drop(columns="Activity")
y_test = test_df["Activity"]

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
y_encoded_test = label_encoder.fit_transform(y_test)

X_test.columns = [
    re.sub(r"[^A-Za-z0-9_]+", "_", col)  # Replace non-alphanumeric/underscore chars with "_"
    for col in X_test.columns
]

features = [feature for feature in X.columns]

# GBM

In [10]:
gb_clf = GradientBoostingClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=7,
    random_state=42
)
gb_clf.fit(X, y_encoded)

GradientBoostingClassifier(learning_rate=0.05, max_depth=7, n_estimators=500,
                           random_state=42)

In [ ]:
# Make predictions on the test set
y_pred = gb_clf.predict(X_test)

# Evaluate accuracy
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_encoded_test, y_pred)
print(f"Test set accuracy: {accuracy:.4f}")


Test set accuracy: 0.9155


# LdaBoost

In [ ]:
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.tree import DecisionTreeRegressor

def softmax(F):
    """Computes the softmax row by row.
       F: matrix of logits of shape (n_samples, n_classes)
    """
    expF = np.exp(F - np.max(F, axis=1, keepdims=True))
    return expF / np.sum(expF, axis=1, keepdims=True)

class CustomMulticlassGradientBoostingClassifier:
    def __init__(self, n_estimators=10, learning_rate=0.1, max_depth=3):
        """
        n_estimators: number of boosting iterations
        learning_rate: learning rate
        max_depth: maximum depth of individual base trees
        """
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        
        # Lists to store the trees for each iteration:
        # self.estimators[m] is a list of regressors, one for each class, at iteration m
        self.estimators = []
        # Store the LDA transformation used at each iteration
        self.lda_transforms = []
        # Initial logit (vector per class) based on prior probabilities
        self.initial_logit = None
        self.classes_ = None

    def fit(self, X, y):
        """
        Trains the model on data X and multiclass target y.
        """
        n_samples = X.shape[0]
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)
        
        # One-hot encoding of the target
        one_hot_y = np.eye(n_classes)[np.searchsorted(self.classes_, y)]
        
        # Compute prior probabilities for each class and the initial logit
        prior = np.mean(one_hot_y, axis=0)
        # Avoid log(0) issues
        prior = np.clip(prior, 1e-5, 1-1e-5)
        self.initial_logit = np.log(prior)  # vector of length n_classes
        
        # Initialize the F matrix (logits) for each sample: same initial logit for all
        F = np.tile(self.initial_logit, (n_samples, 1))
        
        for m in range(self.n_estimators):
            if m == 0:
                # First iteration: LDA with original target (multiclass)
                lda = LinearDiscriminantAnalysis(n_components=None)
                X_lda = lda.fit_transform(X, y)
            else:
                # Compute current probabilities via softmax
                p = softmax(F)
                # Pseudo-residuals: difference between one-hot encoding and current probabilities
                residuals = one_hot_y - p  # shape (n_samples, n_classes)
                # To use LDA: define for each sample the label of the largest residual
                # (the class where the model underestimates the most)
                labels = np.argmax(residuals, axis=1)
                lda = LinearDiscriminantAnalysis(n_components=None)
                X_lda = lda.fit_transform(X, labels)
            
            # Save the current LDA transformation
            self.lda_transforms.append(lda)
            
            # Recalculate residuals based on current logits
            p = softmax(F)
            residuals = one_hot_y - p  # shape (n_samples, n_classes)
            
            # List of regressors for iteration m (one for each class)
            estimators_m = []
            # For each class, train a regressor to predict the residual for that class
            for k in range(n_classes):
                reg = DecisionTreeRegressor(max_depth=self.max_depth)
                reg.fit(X_lda, residuals[:, k])
                estimators_m.append(reg)
                # Update the logit for class k
                update = reg.predict(X_lda)
                F[:, k] += self.learning_rate * update
            
            self.estimators.append(estimators_m)
        
        return self

    def predict_proba(self, X):
        """
        Returns the predicted probabilities for each class.
        """
        n_samples = X.shape[0]
        n_classes = len(self.classes_)
        # Initialize F with the initial logit for each sample
        F = np.tile(self.initial_logit, (n_samples, 1))
        
        # Sequentially apply transformations and updates for each iteration
        for lda, estimators_m in zip(self.lda_transforms, self.estimators):
            X_lda = lda.transform(X)
            for k, reg in enumerate(estimators_m):
                F[:, k] += self.learning_rate * reg.predict(X_lda)
        
        # Compute probabilities via softmax
        return softmax(F)

    def predict(self, X):
        """
        Returns the predicted classes (the class with the highest probability).
        """
        proba = self.predict_proba(X)
        class_indices = np.argmax(proba, axis=1)
        return self.classes_[class_indices]

In [13]:
model = CustomMulticlassGradientBoostingClassifier(n_estimators=1150, learning_rate=0.27871532239639646, max_depth=2)

# X = X.to_numpy()
model.fit(X, y_encoded)

In [14]:
# Evaluate accuracy
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test.to_numpy())
accuracy = accuracy_score(y_encoded_test, y_pred)
print(f"Test set accuracy: {accuracy:.4f}")

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearDiscriminantAnalysis was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearDiscriminantAnalysis was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearDiscriminantAnalysis was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearDiscriminantAnalysis was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.frame

Test set accuracy: 0.9528


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearDiscriminantAnalysis was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearDiscriminantAnalysis was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearDiscriminantAnalysis was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearDiscriminantAnalysis was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.frame

# PCA+GBM and LDA+GBM

In [ ]:
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import Any, Dict

from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

# ---------------------------------------------------------------------
# DuoBoostCV — evaluates LDA+GB and PCA+GB, with CV and external test
# ---------------------------------------------------------------------
@dataclass
class DuoBoostCV:
    """Compares LDA+GradientBoost vs PCA+GradientBoost with cross-validation
    and optionally evaluates accuracy on an external dataset.

    Main Methods
    ------------
    fit(X, y, **cv_kwargs) -> pd.DataFrame
        Performs cross-validation on the training data.

    evaluate_external(X_train, y_train, X_test, y_test, metric=accuracy_score)
        Trains both pipelines on (X_train, y_train) and computes the metric
        (accuracy by default) on (X_test, y_test).
    """

    seed: int = 42
    _models: Dict[str, Any] = field(init=False, repr=False)

    # -----------------------------------------------------------
    # Build the models based on the number of features
    # -----------------------------------------------------------
    def _build_models(self, n_features: int) -> Dict[str, Any]:
        half_components = max(1, n_features // 2)
        return {
            "PCA + Gradient Boost": Pipeline(
                [
                    ("pca", PCA(n_components=half_components, random_state=self.seed)),
                    ("gb", GradientBoostingClassifier(random_state=self.seed)),
                ]
            ),
            "LDA + Gradient Boost": Pipeline(
                [
                    ("lda", LDA(n_components=None)),
                    ("gb", GradientBoostingClassifier(random_state=self.seed)),
                ]
            ),
        }

    # -----------------------------------------------------------
    # Cross-validation on the training data
    # -----------------------------------------------------------
    def fit(self, X, y, **cv_kwargs) -> pd.DataFrame:
        X = np.asarray(X)
        y = np.asarray(y)

        cv_kwargs = cv_kwargs.copy()
        cv_kwargs.setdefault("scoring", "accuracy")

        self._models = self._build_models(X.shape[1])
        scores: Dict[str, Dict[str, float]] = {}

        for name, model in self._models.items():
            cv_res = cross_validate(model, X, y, **cv_kwargs)

            # <<< dynamically find the metric >>>
            metric_key = next(k for k in cv_res if k.startswith("test_"))
            metric_name = metric_key.replace("test_", "")

            scores[name] = {
                f"mean_{metric_name}": cv_res[metric_key].mean(),
                f"std_{metric_name}":  cv_res[metric_key].std(),
            }

        self.results_ = pd.DataFrame(scores).T.sort_values(
            by=f"mean_{metric_name}", ascending=False
        )
        return self.results_

    # -----------------------------------------------------------
    # Evaluation on an external dataset
    # -----------------------------------------------------------
    def evaluate_external(
        self,
        X_train,
        y_train,
        X_test,
        y_test,
        metric=accuracy_score,
    ) -> pd.DataFrame:
        """Trains each pipeline on the training set and computes the metric
        on the external test set.

        Parameters
        ----------
        X_train, y_train : array-like
            Training data.
        X_test, y_test : array-like
            Test data.
        metric : callable, default=accuracy_score
            Function that takes (y_true, y_pred) and returns a float.

        Returns
        -------
        pd.DataFrame
            DataFrame containing the performance on the test set.
        """
        X_train = np.asarray(X_train)
        y_train = np.asarray(y_train)
        X_test = np.asarray(X_test)
        y_test = np.asarray(y_test)

        self._models = self._build_models(X_train.shape[1])

        scores = {}
        for name, model in self._models.items():
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            scores[name] = metric(y_test, y_pred)

        metric_name = metric.__name__.replace("_score", "")
        self.external_results_ = (
            pd.Series(scores, name=f"test_{metric_name}")
            .to_frame()
            .sort_values(by=f"test_{metric_name}", ascending=False)
        )
        return self.external_results_

    # Convenience method
    def __call__(self, X, y, **cv_kwargs) -> pd.DataFrame:
        return self.fit(X, y, **cv_kwargs)

    def __repr__(self) -> str:  # pragma: no cover
        rep = f"{self.__class__.__name__}(seed={self.seed})"
        if hasattr(self, "results_"):
            rep += f"\nCV results:\n{self.results_.round(4)}"
        if hasattr(self, "external_results_"):
            rep += f"\nExternal test results:\n{self.external_results_.round(4)}"
        return rep

# ---------------------------------------------------------------------
# Usage Example
# ---------------------------------------------------------------------
trainer = DuoBoostCV(seed=42)

cv_results = trainer(X, y_encoded, cv=10, scoring="accuracy", n_jobs=-1)
print(cv_results)

test_results = trainer.evaluate_external(X, y_encoded, X_test, y_encoded_test)
print(test_results)


                      mean_score  std_score
LDA + Gradient Boost    0.946144   0.045751
PCA + Gradient Boost    0.892421   0.049878
                      test_accuracy
LDA + Gradient Boost       0.953512
PCA + Gradient Boost       0.908042
